In [0]:
from pyspark.sql.functions import col,expr

# Perform row count checks
df = spark.table("main.ecommerce.fact_sales")

row_count =df.count()
#display(row_count)
#------------------------------------------

# Validate nulls in key columns
null_count = df.filter(col("order_id").isNull() | col("order_item_id").isNull() | col("customer_id").isNull() | col("product_id").isNull() | col("order_date").isNull() | col("revenue").isNull()).count()
#display(null_count)
#-----------------------------------------

# Perform revenue sanity checks
invalid_revenue_cnt = df.filter((col("revenue") <= 0) | (col("revenue") != expr("price + freight_value"))).count()

#display(invalid_revenue_cnt)
#----------------------------------------

if row_count == 0:
        raise Exception("Validation Failed: fact_table has zero rows")
else:
    if null_count > 0:
        raise Exception("Validation Failed: Null values found in key columns")
    else:
        if invalid_revenue_cnt > 0:
            raise Exception("Validation Failed: Revenue sanity check failed")
        else:
            print("All data quality checks passed successfully")

# Fail the pipeline if validations fail